In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("evalj2").getOrCreate()
sc = spark.sparkContext

print("Version de Spark :", spark.version)
print("Master           :", sc.master)
print("Parallélisme     :", sc.defaultParallelism)

Version de Spark : 4.2.0
Master           : local[*]
Parallélisme     : 8


In [6]:
clients_churn = spark.read.csv("data/clients_churn.csv",header=True,inferSchema=True)

In [7]:
clients_churn.show(10)

+---------+---------------+---------------+------------+---------------+----------+-----+
|id_client|anciennete_mois|nb_commandes_6m|panier_moyen|nb_reclamations|abonnement|parti|
+---------+---------------+---------------+------------+---------------+----------+-----+
|        1|             56|             18|          96|              2|  standard|    0|
|        2|             47|              7|         210|              3|  standard|    1|
|        3|             56|              6|         119|              2|   premium|    0|
|        4|             53|             11|         163|              3|  standard|    1|
|        5|             67|             11|         153|              0|   premium|    0|
|        6|             28|              1|          88|              4|  standard|    0|
|        7|             20|              2|         112|              5|  standard|    1|
|        8|             13|              4|         240|              0|  standard|    0|
|        9

In [8]:
train, test = clients_churn.randomSplit([0.8, 0.2], seed=42)

In [10]:
train.count(),clients_churn.count()

(1642, 2000)

In [11]:
from pyspark.ml.classification import RandomForestClassifier

In [13]:
from pyspark.ml.feature import StringIndexer, VectorAssembler
indexeur = StringIndexer(inputCol="abonnement", outputCol="abo_idx")  

In [14]:
indexeur_entraine = indexeur.fit(clients_churn) 

In [17]:
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml import Pipeline, PipelineModel

assembleur = VectorAssembler(                                        # un transformer
    inputCols=["anciennete_mois", "nb_commandes_6m", "panier_moyen", "nb_reclamations","abo_idx"], outputCol="features")

In [18]:

pipeline = Pipeline(stages=[indexeur_entraine, assembleur, RandomForestClassifier(labelCol="parti", featuresCol="features",
                               numTrees=50, maxDepth=5, seed=42)])

In [19]:
modele_eval = pipeline.fit(train)

In [21]:
prediction = modele_eval.transform(test)

In [22]:
prediction.show()

+---------+---------------+---------------+------------+---------------+----------+-----+-------+--------------------+--------------------+--------------------+----------+
|id_client|anciennete_mois|nb_commandes_6m|panier_moyen|nb_reclamations|abonnement|parti|abo_idx|            features|       rawPrediction|         probability|prediction|
+---------+---------------+---------------+------------+---------------+----------+-----+-------+--------------------+--------------------+--------------------+----------+
|        3|             56|              6|         119|              2|   premium|    0|    1.0|[56.0,6.0,119.0,2...|[46.3394065423038...|[0.92678813084607...|       0.0|
|        7|             20|              2|         112|              5|  standard|    1|    0.0|[20.0,2.0,112.0,5...|[4.98441573072393...|[0.09968831461447...|       1.0|
|        9|             32|             11|         101|              0|  standard|    0|    0.0|[32.0,11.0,101.0,...|[46.3697063339814...|[